# 24 — Backend API contract helpers (V3, research only)

Reusable, notebook-owned query logic for the thin local FastAPI transport in `backend/api/` (task 019):

- **FUNCTIONS 1/3** — error families → HTTP status, strict JSON bodies (64 KiB, UTF-8, no duplicate keys, no NaN/Infinity), message sanitising, exact body contracts, mapping of notebook 13/14/22 answers.
- **FUNCTIONS 2/3** — read-only geometry index over the ACCEPTED notebook-19 universe (3105 qualified DOH segments) and bounded bbox / page / segment queries (EPSG:4326 GeoJSON).
- **FUNCTIONS 3/3** — aggregate evaluation summaries (accepted 21 and 23), spatial-history period status (2026+ unsupported, never zero) and composed metadata.

No model, no fit, no feature path, no network. Predictions come only from the accepted notebook 22 (spatial) and 13 (province, V2 compatibility); history only from the accepted notebook 14. The API executes these FUNCTIONS only after the manager pins this notebook's FUNCTIONS hash in `workflow/api/input-pins-v3.json`; until then the default server stays unavailable (fail closed).


In [ ]:
# PARAMETERS: repository locations only (no output is written by this notebook).
import os
from pathlib import Path


def _default_backend(c):
    for p in (c, *c.parents):
        if (p / 'config' / 'backend_api_v3.json').is_file():
            return p
        if (p / 'backend' / 'config' / 'backend_api_v3.json').is_file():
            return p / 'backend'
    raise FileNotFoundError('backend root not found; set ROADAWARE_BACKEND_ROOT')


BACKEND = Path(os.environ.get('ROADAWARE_BACKEND_ROOT') or _default_backend(Path.cwd())).resolve()
REPO = BACKEND.parent
CONFIG_PATH = BACKEND / 'config' / 'backend_api_v3.json'


In [ ]:
# FUNCTIONS (1/3): API error families and HTTP status mapping, strict JSON body parsing (bounded, UTF-8, no duplicate keys, no NaN/Infinity,
# no overflowing floats), message sanitising and the exact top-level body contracts. Pure logic shared by backend/api (thin transport) and the
# tests; no file I/O, no model, no network. Row-level request validation stays in the accepted notebooks 13/14/22.
import datetime as dt
import json
import math
import re
import types

import numpy as np
import pandas as pd

NL = chr(10)
API_VERSION = 'roadaware-api-v3.0'
FAMILY_STATUS = {'invalid': 400, 'coverage': 404, 'not_found': 404, 'method': 405, 'too_large': 413, 'media_type': 415, 'unavailable': 503,
                 'internal': 500}
GENERIC_MESSAGE = {'unavailable': 'a required accepted artifact is unavailable or failed verification; details are in the local server log only',
                   'internal': 'internal error; the request was not answered (no partial or zero result)'}
PATH_RE = re.compile(r'(?:[A-Za-z]:\\|~/|/)(?:[^\s\'"(),;<>]+[/\\])+[^\s\'"(),;<>]*')
MAX_MESSAGE = 300
MAX_ERROR_ITEMS = 128
RESEARCH_FLAGS = {'research_only': True, 'production_ready': False}


class ApiError(Exception):
    """A refused request or an unavailable component. family selects the HTTP status; message is sanitised before it is returned."""

    def __init__(self, code, family, message, rows=None, errors=None):
        super().__init__(str(code) + ': ' + str(message))
        if family not in FAMILY_STATUS:
            raise ValueError('unknown error family ' + repr(family))
        self.code, self.family, self.message, self.status = str(code), family, str(message), FAMILY_STATUS[family]
        self.rows, self.errors = rows, errors


def sanitize_message(message, family):
    """Unavailable/internal details never leave the server (paths, hashes, tracebacks); other messages lose anything path-like and are bounded."""
    if family in GENERIC_MESSAGE:
        return GENERIC_MESSAGE[family]
    return PATH_RE.sub('<redacted>', str(message))[:MAX_MESSAGE]


def _clean_item(e):
    out = {}
    for k in ('index', 'field', 'code', 'message'):
        v = e.get(k) if isinstance(e, dict) else None
        if k == 'index':
            out[k] = int(v) if isinstance(v, (int, np.integer)) and not isinstance(v, bool) else None
        else:
            out[k] = None if v is None else sanitize_message(v, 'invalid')[:120]
    return out


def error_body(e):
    err = {'code': e.code, 'family': e.family, 'message': sanitize_message(e.message, e.family)}
    if e.rows is not None:
        err['rows'] = [int(r) for r in list(e.rows)[:MAX_ERROR_ITEMS]]
    if e.errors is not None:
        err['errors'] = [_clean_item(x) for x in list(e.errors)[:MAX_ERROR_ITEMS]]
    return {'ok': False, 'api_version': API_VERSION, 'production_ready': False, 'error': err}


def ok_body(payload):
    out = {'ok': True, 'api_version': API_VERSION, 'production_ready': False}
    out.update(payload)
    out['production_ready'] = False
    return out


def check_finite(obj, depth=0):
    """Strict JSON value check: dict(str keys)/list/str/int/finite float/bool/None only. Raises ValueError (never silently nulls)."""
    if depth > 64:
        raise ValueError('JSON nesting too deep')
    if isinstance(obj, (dict, types.MappingProxyType)):
        for k, v in obj.items():
            if not isinstance(k, str):
                raise ValueError('non-string JSON key')
            check_finite(v, depth + 1)
    elif isinstance(obj, (list, tuple)):
        for v in obj:
            check_finite(v, depth + 1)
    elif isinstance(obj, float):
        if not math.isfinite(obj):
            raise ValueError('non-finite float in a response')
    elif obj is None or isinstance(obj, (str, bool, int)):
        return obj
    else:
        raise ValueError('non-JSON value of type ' + type(obj).__name__)
    return obj


def strict_dumps(obj):
    check_finite(obj)
    return json.dumps(obj, ensure_ascii=False, allow_nan=False, separators=(',', ':'))


# ---------- strict request bodies ----------
def _pairs(pairs):
    d = {}
    for k, v in pairs:
        if k in d:
            raise ApiError('invalid_json', 'invalid', 'duplicate JSON object key')
        d[k] = v
    return d


def _constant(c):
    raise ApiError('invalid_json', 'invalid', 'non-finite JSON constant refused (NaN/Infinity)')


def _float(s):
    v = float(s)
    if not math.isfinite(v):
        raise ApiError('invalid_json', 'invalid', 'number overflows to a non-finite float')
    return v


def parse_json_body(raw, max_bytes):
    if not isinstance(raw, (bytes, bytearray)):
        raise ApiError('invalid_json', 'invalid', 'request body must be bytes')
    if len(raw) > int(max_bytes):
        raise ApiError('body_too_large', 'too_large', 'request body exceeds ' + str(int(max_bytes)) + ' bytes')
    if not raw:
        raise ApiError('invalid_json', 'invalid', 'empty request body')
    try:
        text = bytes(raw).decode('utf-8')
    except UnicodeDecodeError:
        raise ApiError('invalid_json', 'invalid', 'request body must be UTF-8 JSON')
    try:
        return json.loads(text, object_pairs_hook=_pairs, parse_constant=_constant, parse_float=_float)
    except ApiError:
        raise
    except (ValueError, RecursionError):
        raise ApiError('invalid_json', 'invalid', 'malformed JSON body')


def exact_object(obj, keys, label):
    if type(obj) is not dict:
        raise ApiError('invalid_request', 'invalid', label + ' body must be a JSON object')
    if set(obj) != set(keys):
        raise ApiError('invalid_fields', 'invalid', label + ' body fields must be exactly ' + str(sorted(keys)))
    return obj


def spatial_prediction_body(obj):
    """{"unit": "daily"|"6h", "requests": [...]}; unit/rows are validated by the accepted notebook-22 predict_batch (no coercion here)."""
    o = exact_object(obj, ('unit', 'requests'), 'spatial prediction')
    if type(o['requests']) is not list:
        raise ApiError('invalid_request', 'invalid', 'requests must be a JSON array of objects')
    return o['unit'], o['requests']


def province_prediction_body(obj):
    """{"requests": [...]} for the preserved V2 province endpoint; rows are validated by the accepted notebook-13 validate_requests."""
    o = exact_object(obj, ('requests',), 'province prediction')
    if type(o['requests']) is not list:
        raise ApiError('invalid_request', 'invalid', 'requests must be a JSON array of objects')
    return o['requests']


def spatial_history_body(obj):
    if type(obj) is not dict:
        raise ApiError('invalid_request', 'invalid', 'spatial history body must be a JSON object')
    return obj


# ---------- component answers -> API answers ----------
def spatial_answer(res):
    """Notebook-22 response mapping: ok -> 200 body; its stable error families (invalid/coverage/unavailable) keep code and offending rows."""
    if not isinstance(res, dict):
        raise ApiError('prediction_invalid', 'unavailable', 'notebook-22 response is not an object')
    if res.get('ok') is True:
        return ok_body({k: v for k, v in res.items() if k != 'ok'})
    e = res.get('error') or {}
    fam = e.get('family') if e.get('family') in ('invalid', 'coverage', 'unavailable') else 'unavailable'
    raise ApiError(e.get('code', 'artifact_unavailable'), fam, e.get('message', ''), rows=e.get('rows') if fam != 'unavailable' else None)


COVERAGE_QUERY_CODES = ('unknown_segment', 'unknown_province')


def catalog_query_error(errors):
    """Notebook-14 CatalogQueryError.errors -> one ApiError. Only-unknown-id errors are coverage (404), anything else invalid (400)."""
    errs = list(errors or [])
    codes = {str(e.get('code')) for e in errs if isinstance(e, dict)}
    if codes and codes <= set(COVERAGE_QUERY_CODES):
        return ApiError(sorted(codes)[0], 'coverage', 'identifier outside the accepted catalog universe (no history, never zero)', errors=errs)
    return ApiError('invalid_request', 'invalid', 'history request rejected by the accepted catalog contract', errors=errs)


def province_request_error(errors):
    """Notebook-13 RequestValidationError.errors -> invalid ApiError; per-row messages are replaced by codes only (no input echo)."""
    errs = [{'index': e.get('index'), 'field': e.get('field'), 'code': e.get('code'), 'message': None} for e in list(errors or []) if isinstance(e, dict)]
    return ApiError('invalid_request', 'invalid', 'province prediction request rejected by the accepted notebook-13 contract', errors=errs)


def strict_query(params, allowed, label):
    """params: list of (key, value) pairs of the query string. Unknown or repeated keys are refused; returns {key: str}."""
    out = {}
    for k, v in params:
        if k not in allowed:
            raise ApiError('invalid_fields', 'invalid', label + ' query parameters must be within ' + str(sorted(allowed)))
        if k in out:
            raise ApiError('invalid_fields', 'invalid', 'repeated query parameter ' + str(k)[:40])
        if not isinstance(v, str) or len(v) > 400:
            raise ApiError('invalid_value', 'invalid', 'query parameter ' + str(k)[:40] + ' too long')
        out[k] = v
    return out


INT_RE = re.compile(r'[0-9]{1,6}')


def parse_int_text(v, field, lo, hi):
    if not isinstance(v, str) or not INT_RE.fullmatch(v):
        raise ApiError('invalid_value', 'invalid', field + ' must be a plain non-negative integer')
    n = int(v)
    if not lo <= n <= hi:
        raise ApiError('invalid_value', 'invalid', field + ' must be in [' + str(lo) + ', ' + str(hi) + ']')
    return n


def province_history_request(q, max_top_k):
    """GET /api/v1/history query -> the notebook-14 province request dict (types exact; 14 validates ranges/keys)."""
    req = {}
    for k in ('province_key', 'start_month', 'end_month'):
        if k in q:
            req[k] = q[k]
    if 'top_k' in q:
        req['top_k'] = parse_int_text(q['top_k'], 'top_k', 1, int(max_top_k))
    if 'categories' in q:
        req['categories'] = q['categories'].split(',') if q['categories'] else []
    return req


In [ ]:
# FUNCTIONS (2/3): read-only geometry index over the ACCEPTED notebook-19 universe (universe_geometry.geojson + universe_segments.csv, both
# hash-verified by the caller before parsing) and bounded bbox/page/segment queries returning EPSG:4326 GeoJSON. No road rebuilding, no snapping.
GEOMETRY_NOTE = ('current qualified DOH 2000 m segment snapshot (EPSG:4326 lon/lat, RFC 7946 order) applied retrospectively; NOT a routing graph '
                 'and NOT the true historical road network of earlier years')
OTHER_ROADS_RULE = ('roads outside the 3105 qualified DOH segments are unsupported: render them grey / "no data"; the API never returns a '
                    'prediction, a zero or a lambda of 0 for them and never snaps a location to a segment')
SEGMENT_ID_RE = re.compile(r'[A-Za-z0-9:._-]{1,200}')
BBOX_NUM_RE = re.compile(r'-?[0-9]{1,3}(\.[0-9]{1,12})?')


def _req(ok, msg):
    if not ok:
        raise ApiError('artifact_inconsistent', 'unavailable', msg)


def build_geometry_index(fc, seg_rows, expected_ids, limits):
    """fc: parsed FeatureCollection; seg_rows: universe_segments.csv as a str DataFrame; expected_ids: the notebook-22 service universe (or None).
    Fails closed (unavailable) unless ids are unique non-empty strings, geojson ids == csv ids (== expected), every geometry is a finite 2-D
    LineString inside the configured lon/lat envelope, properties are exactly {route_code} and equal the csv, and lengths are finite > 0."""
    lon_lo, lon_hi = (float(x) for x in limits['lon_envelope'])
    lat_lo, lat_hi = (float(x) for x in limits['lat_envelope'])
    _req(isinstance(fc, dict) and fc.get('type') == 'FeatureCollection' and isinstance(fc.get('features'), list), 'geometry is not a FeatureCollection')
    need = ('segment_id', 'route_code', 'geom_length_m', 'province_code')
    _req(isinstance(seg_rows, pd.DataFrame) and all(c in seg_rows.columns for c in need), 'universe_segments.csv lacks ' + str(need))
    ids, coords, routes = [], {}, {}
    for f in fc['features']:
        _req(isinstance(f, dict) and f.get('type') == 'Feature', 'geometry member is not a Feature')
        sid = f.get('id')
        _req(isinstance(sid, str) and SEGMENT_ID_RE.fullmatch(sid) is not None, 'feature id is not a stable segment id string')
        g = f.get('geometry') or {}
        _req(g.get('type') == 'LineString', 'segment geometry is not a LineString')
        c = np.asarray(g.get('coordinates', []), dtype=float)
        _req(c.ndim == 2 and c.shape[0] >= 2 and c.shape[1] == 2 and np.isfinite(c).all(), 'segment geometry is not a finite 2-D LineString')
        _req(bool((c[:, 0] >= lon_lo).all() and (c[:, 0] <= lon_hi).all() and (c[:, 1] >= lat_lo).all() and (c[:, 1] <= lat_hi).all()),
             'segment geometry outside the lon/lat envelope (axis order or CRS wrong)')
        p = f.get('properties')
        _req(isinstance(p, dict) and set(p) == {'route_code'} and isinstance(p['route_code'], str), 'segment properties must be exactly {route_code}')
        ids.append(sid)
        coords[sid] = c
        routes[sid] = p['route_code']
    _req(len(ids) == len(set(ids)) and len(ids) > 0, 'geometry segment ids are empty or not unique')
    S = seg_rows.astype(str)
    _req(not S['segment_id'].duplicated().any(), 'universe_segments.csv segment ids not unique')
    _req(set(S['segment_id']) == set(ids), 'geometry ids differ from universe_segments.csv ids')
    if expected_ids is not None:
        _req(set(expected_ids) == set(ids) and len(list(expected_ids)) == len(ids), 'geometry ids differ from the notebook-22 service universe')
    S = S.set_index('segment_id')
    _req(all(S.at[s, 'route_code'] == routes[s] for s in ids), 'geometry route_code differs from universe_segments.csv')
    order = sorted(ids)
    length = pd.to_numeric(S.loc[order, 'geom_length_m'], errors='coerce').to_numpy(np.float64)
    _req(bool(np.isfinite(length).all() and (length > 0).all()), 'segment lengths must be finite and positive')
    prov = pd.to_numeric(S.loc[order, 'province_code'], errors='coerce').to_numpy(np.float64)
    bbox = np.array([[coords[s][:, 0].min(), coords[s][:, 1].min(), coords[s][:, 0].max(), coords[s][:, 1].max()] for s in order], dtype=np.float64)
    bbox.setflags(write=False)
    return types.MappingProxyType({
        'ids': tuple(order), 'pos': types.MappingProxyType({s: i for i, s in enumerate(order)}),
        'coords': tuple(tuple((float(x), float(y)) for x, y in coords[s]) for s in order), 'route': tuple(routes[s] for s in order),
        'km': tuple(float(v) / 1000.0 for v in length),
        'province_code': tuple(int(v) if math.isfinite(v) and float(v).is_integer() else None for v in prov), 'bbox': bbox,
        'extent': [float(bbox[:, 0].min()), float(bbox[:, 1].min()), float(bbox[:, 2].max()), float(bbox[:, 3].max())], 'n': len(order)})


def parse_bbox(text, limits):
    """'west,south,east,north' decimal degrees (no exponent/NaN/Infinity), finite, west<east, south<north, inside lon/lat, span bounded."""
    if not isinstance(text, str):
        raise ApiError('invalid_bbox', 'invalid', 'bbox is required as west,south,east,north')
    parts = text.split(',')
    if len(parts) != 4 or not all(BBOX_NUM_RE.fullmatch(p) for p in parts):
        raise ApiError('invalid_bbox', 'invalid', 'bbox must be four plain decimal numbers west,south,east,north')
    w, s, e, n = (float(p) for p in parts)
    if not all(math.isfinite(v) for v in (w, s, e, n)) or not (-180 <= w <= 180 and -180 <= e <= 180 and -90 <= s <= 90 and -90 <= n <= 90):
        raise ApiError('invalid_bbox', 'invalid', 'bbox outside lon [-180,180] / lat [-90,90]')
    if not (w < e and s < n):
        raise ApiError('invalid_bbox', 'invalid', 'bbox needs west < east and south < north (no antimeridian wrap)')
    mx = float(limits['max_bbox_span_deg'])
    if e - w > mx or n - s > mx:
        raise ApiError('bbox_too_large', 'invalid', 'bbox span must be at most ' + str(mx) + ' degrees per axis')
    return w, s, e, n


def parse_page(q, limits):
    lim = parse_int_text(q['limit'], 'limit', 1, int(limits['max_page_size'])) if 'limit' in q else int(limits['default_page_size'])
    cur = parse_int_text(q['cursor'], 'cursor', 0, 10 ** 6) if 'cursor' in q else 0
    return lim, cur


def segment_feature(index, i):
    sid = index['ids'][i]
    return {'type': 'Feature', 'id': sid, 'geometry': {'type': 'LineString', 'coordinates': [list(p) for p in index['coords'][i]]},
            'properties': {'segment_id': sid, 'route_code': index['route'][i], 'segment_km': index['km'][i], 'province_code': index['province_code'][i],
                           'coverage': 'supported_qualified_doh_segment'}}


def query_segments(index, bbox, limit, cursor, snapshot):
    """Segments whose envelope intersects bbox, ordered by segment_id ascending; cursor = offset into that stable order (index is immutable)."""
    w, s, e, n = bbox
    B = index['bbox']
    hit = np.flatnonzero((B[:, 0] <= e) & (B[:, 2] >= w) & (B[:, 1] <= n) & (B[:, 3] >= s))
    total = int(len(hit))
    if cursor > total:
        raise ApiError('invalid_cursor', 'invalid', 'cursor beyond the matched segments (' + str(total) + ')')
    page = hit[cursor:cursor + limit]
    nxt = cursor + limit if cursor + limit < total else None
    return ok_body({'type': 'FeatureCollection', 'bbox': [w, s, e, n], 'features': [segment_feature(index, int(i)) for i in page],
                    'page': {'limit': int(limit), 'cursor': str(cursor), 'next_cursor': None if nxt is None else str(nxt), 'returned': int(len(page)),
                             'total_matched': total, 'order': 'segment_id ascending (stable for this geometry snapshot)'},
                    'coordinate_reference': 'EPSG:4326 longitude, latitude (RFC 7946)', 'geometry_snapshot': snapshot,
                    'notes': [GEOMETRY_NOTE, OTHER_ROADS_RULE]})


def get_segment(index, segment_id, unsupported_ids, snapshot):
    if not isinstance(segment_id, str) or SEGMENT_ID_RE.fullmatch(segment_id) is None:
        raise ApiError('invalid_value', 'invalid', 'segment_id must be a stable segment id string')
    i = index['pos'].get(segment_id)
    if i is None:
        code = 'unsupported_segment' if segment_id in unsupported_ids else 'unknown_segment'
        raise ApiError(code, 'coverage', 'segment outside the qualified DOH universe: unsupported (grey), never zero')
    return ok_body({'feature': segment_feature(index, i), 'coordinate_reference': 'EPSG:4326 longitude, latitude (RFC 7946)',
                    'geometry_snapshot': snapshot, 'notes': [GEOMETRY_NOTE, OTHER_ROADS_RULE]})


In [ ]:
# FUNCTIONS (3/3): aggregate-only evaluation summaries (accepted notebook 21 manifest + its manifest-verified pooled/period selected/baseline
# OOF and seen-2024/25 metrics_by_group rows; accepted notebook 23 overall metrics with BOTH the raw
# deviance status (+infinity encoded null) and the original 1e-12 floor audit), spatial-history period status (2026+ is NOT in the accepted
# catalog: unsupported, never an empty zero) and the composed metadata. Inputs are already hash-verified by the caller; nothing here reads rows.
METRIC_NUM = ('n_segments', 'n_positive_cells', 'n_raw_records', 'actual_total', 'pred_total', 'mean_poisson_deviance', 'n_zero_pred_on_positive', 'mae',
              'rmse', 'bias_pred_minus_actual', 'wape', 'pred_to_actual_ratio', 'deviance_floor_audit', 'deviance_floor_audit_floor',
              'deviance_floor_audit_n_floored', 'deviance_floor_audit_floored_contribution', 'deviance_floor_audit_floored_share')
DIAG_LIMITS = [
    'January-September 2026 RAW OBSERVED-RECORD diagnostic of the official MOT snapshot; NOT a strict untouched test (2026 was audited before) and '
    'NOT a validated nationwide or complete-September evaluation.',
    'Only observed POSITIVE eligible linked cells are scored (outcome-conditioned): zero labels, full calendar x segment population and unique '
    'physical-event identity (no ACC_CODE) are UNVERIFIED; missing cells stay unknown, never zero.',
    'Provider checksum NOT verified (the CKAN hash differs from the downloaded file MD5 and the saved metadata does not name its algorithm); the local '
    'SHA-256 snapshot pin matches.',
    'Raw strong-baseline deviance is +infinity where it predicts 0 on a positive cell: encoded as null with deviance_status; the original 1e-12 floor '
    'audit is reported separately with its floored counts.']


def _num(v):
    if v is None or v == '':
        return None
    x = float(v)
    if not math.isfinite(x):
        return None
    return int(x) if x.is_integer() and abs(x) < 2 ** 53 and not isinstance(v, float) and '.' not in str(v) else x


def diagnostics_2026_summary(record, manifest, metric_rows):
    """record: manager-accepted 23 record; manifest: hash-verified 23 manifest; metric_rows: dicts of the hash-verified metrics.csv."""
    A = record['acceptance']
    _req(manifest.get('status') == 'complete' and manifest.get('research_only') is True and manifest.get('production_ready') is False,
         '23 manifest not complete/research-only')
    _req(manifest.get('checks_failed') == 0 and manifest.get('evaluation_available') is True, '23 manifest has failed checks or no evaluation')
    C = manifest.get('counts') or {}
    _req(C.get('eligible_records') == A['eligible_raw_records'] and C.get('raw_records') == A['raw_rows']
         and (C.get('positive_cells') or {}) == {'daily': A['daily_positive_cells'], '6h': A['six_hour_positive_cells']}, '23 manifest counts differ from the accepted record')
    overall = []
    for r in metric_rows:
        if r.get('dim') != 'overall':
            continue
        row = {'unit': r['unit'], 'method': r['method'], 'deviance_status': r['deviance_status']}
        row.update({k: _num(r.get(k)) for k in METRIC_NUM})
        if row['deviance_status'] != 'finite':
            row['mean_poisson_deviance'] = None
        overall.append(row)
    _req(sorted((o['unit'], o['method']) for o in overall) == [('6h', 'selected'), ('6h', 'strong_baseline'), ('daily', 'selected'), ('daily', 'strong_baseline')],
         '23 overall metrics must hold selected + strong_baseline for both units')
    by = {(o['unit'], o['method']): o for o in overall}
    mae_better = {u: ('strong_baseline' if by[(u, 'strong_baseline')]['mae'] < by[(u, 'selected')]['mae'] else 'selected') for u in ('daily', '6h')}
    return {'status': 'available_diagnostic_aggregates_only', 'kind': record['kind'], 'version': manifest.get('version'),
            'record_meaning': manifest.get('record_meaning'), 'window': {'first_date': A['eligible_first_date'], 'last_date': A['eligible_last_date']},
            'counts': {'raw_rows_in_snapshot': A['raw_rows'], 'requested_window_rows': A['requested_window_rows'], 'eligible_raw_records': A['eligible_raw_records'],
                       'positive_cells': {'daily': A['daily_positive_cells'], '6h': A['six_hour_positive_cells']},
                       'segments_with_observed_records': A['segments_with_observed_records'], 'recorded_provinces': A['recorded_provinces'],
                       'exclusion_reasons': C.get('reasons')},
            'overall_metrics': overall, 'lower_mae_on_this_conditional_cohort': mae_better,
            'source_checksum_status': record['source_checksum_status'], 'limits': DIAG_LIMITS, 'caveats': list(manifest.get('caveats') or []),
            'runtime': manifest.get('runtime'), **RESEARCH_FLAGS}


EVAL21_SCOPES = {'oof': '2022-23 out-of-fold (the selection period)',
                 'diag': 'PREVIOUSLY SEEN 2024-25 full-population diagnostic of the frozen benchmark artifact (not an untouched test)'}
EVAL21_DIMENSIONS = ('pooled', 'period')
EVAL21_ROLES = ('selected', 'baseline')
EVAL21_HORIZON = {'daily': 'local day (Asia/Bangkok)', '6h': '6 h block (Asia/Bangkok)'}
EVAL21_NUM = ('n', 'zero_cells', 'nonzero_cells', 'total_actual', 'total_pred', 'pred_over_actual', 'mean_poisson_deviance', 'mae', 'rmse', 'bias', 'wape',
              'exposure_segment_km_periods', 'observed_records_per_km_period', 'predicted_records_per_km_period', 'metric_floor_cells',
              'metric_floor_actual_positive_cells', 'metric_floor_actual_records', 'metric_floor_deviance_sum')
EVAL21_MAX_ROWS = 64
EVAL21_COLUMNS = ('n = scored cells (segment x day or x 6 h block); nonzero_cells = cells with >= 1 recorded+linked record; total_actual = recorded+linked '
                  'records; total_pred = sum of predicted expected counts; bias = mean(pred - actual); wape with wape_status; metric_floor_* = the original '
                  'metric-floor audit (cells floored, floored positive cells/records, floored deviance sum) where the method needed it')


def evaluation_21_metrics(manifest, metric_rows):
    """Bounded numeric rows of the hash-verified 21 metrics_by_group.csv: selected + strong baseline (methods must equal the accepted 21 selection),
    scopes oof/diag, dimensions pooled/period only. Values are the CSV numbers (non-finite -> null); nothing is recomputed or re-selected."""
    sel = manifest.get('selected') or {}
    _req(isinstance(sel, dict) and set(sel) == set(EVAL21_HORIZON) and all(isinstance(sel[u], dict) and set(sel[u]) >= set(EVAL21_ROLES) for u in sel),
         '21 manifest selection must name selected + baseline for both units')
    out = []
    for r in metric_rows:
        if r.get('dimension') not in EVAL21_DIMENSIONS or r.get('role') not in EVAL21_ROLES or r.get('scope') not in EVAL21_SCOPES:
            continue
        u = r.get('unit')
        _req(u in sel and r.get('method') == sel[u][r['role']], '21 metrics row method differs from the accepted 21 selection')
        row = {'unit': u, 'horizon': EVAL21_HORIZON[u], 'scope': r['scope'], 'scope_meaning': EVAL21_SCOPES[r['scope']], 'role': r['role'],
               'method': r['method'], 'dimension': r['dimension'], 'group': r.get('group'), 'wape_status': r.get('wape_status') or None}
        row.update({k: _num(r.get(k)) for k in EVAL21_NUM})
        out.append(row)
    keys = [(o['unit'], o['scope'], o['role'], o['dimension'], o['group']) for o in out]
    _req(0 < len(out) <= EVAL21_MAX_ROWS and len(keys) == len(set(keys)), '21 metrics rows missing, duplicated or unbounded')
    _req(all((u, s, ro, 'pooled', 'all') in keys for u in EVAL21_HORIZON for s in EVAL21_SCOPES for ro in EVAL21_ROLES),
         '21 metrics must hold pooled selected + baseline rows for both units and both scopes')
    order = {k: i for i, k in enumerate(list(EVAL21_HORIZON) + list(EVAL21_SCOPES) + list(EVAL21_ROLES) + list(EVAL21_DIMENSIONS))}
    return sorted(out, key=lambda o: (order[o['unit']], order[o['scope']], order[o['dimension']], str(o['group']), order[o['role']]))


def evaluation_21_summary(record, manifest, metric_rows):
    """record: manager-accepted 21 record; manifest: hash-verified 21 manifest; metric_rows: dicts of the manifest-verified metrics_by_group.csv."""
    _req(manifest.get('status') == 'complete' and manifest.get('functions_sha256') == record['functions_sha256'] and not manifest.get('failed_checks'),
         '21 manifest not complete or FUNCTIONS differ from the accepted record')
    return {'status': record['status'], 'version': manifest.get('version'), 'label_meaning': manifest.get('label_meaning'),
            'selected': manifest.get('selected'), 'experiment_lock_sha256': manifest.get('experiment_lock_sha256'),
            'colour_contract_sha256': manifest.get('colour_contract_sha256'), 'checks_passed': len(manifest.get('checks') or []),
            'scope': '2022-23 out-of-fold selection and PREVIOUSLY SEEN 2024-25 full-population diagnostics of the selected policy; no untouched test',
            'metrics': evaluation_21_metrics(manifest, metric_rows), 'metrics_source': 'metrics_by_group.csv (sha256 verified against the accepted 21 manifest)',
            'metrics_columns': EVAL21_COLUMNS,
            'metrics_note': 'selection is unchanged (accepted 21); the 2026 record diagnostic is reported separately under diagnostics_2026 and is outcome-conditioned',
            'report_files': sorted(manifest.get('report_files') or {}), **RESEARCH_FLAGS}


def spatial_history_period(res, first_month, last_month):
    """Wraps a notebook-14 spatial history answer with the explicit period status. History after the accepted catalog end (2026+) or before its
    start is UNSUPPORTED (unknown), never an empty zero."""
    W = res['history_window']
    start_excl, cutoff = W['window_start_local_exclusive'], res['history_cutoff_local']
    data_start, data_end = W['catalog_data_start_local'], W['catalog_data_end_local_exclusive']
    after = cutoff > data_end
    before = start_excl < data_start
    status = 'within_accepted_catalog' if W['window_fully_inside_catalog_data'] else 'partially_unsupported_period'
    return ok_body({'history': res, 'catalog_period': {'first_month': first_month, 'last_month': last_month},
                    'period_status': {'status': status,
                                      'unsupported_after': {'from_local': data_end, 'to_local_inclusive': cutoff} if after else None,
                                      'unsupported_before': {'from_local_exclusive': start_excl, 'to_local': data_start} if before else None,
                                      'note': ('records after ' + last_month + ' (e.g. 2026) are NOT in the accepted catalog 14 and records before ' + first_month +
                                               ' are outside it: those periods are unsupported/unknown, never an empty zero; counts cover the supported part only')}})


def api_metadata(cfg, components, spatial_meta, catalog_summary, province_summary, geometry, evaluation_status):
    return ok_body({
        'service': 'RoadAware local research backend', 'components': components,
        'routes': cfg['routes_doc'], 'limits': cfg['limits'],
        'spatial_runtime': spatial_meta, 'catalog': catalog_summary, 'province_runtime_v2': province_summary,
        'geometry': None if geometry is None else {'segments': geometry['n'], 'extent_lon_lat': geometry['extent'], 'snapshot': cfg['geometry']['snapshot'],
                                                   'note': GEOMETRY_NOTE, 'other_roads': OTHER_ROADS_RULE},
        'evaluation_status': evaluation_status, 'scope_statements': cfg['scope_statements'], **RESEARCH_FLAGS})


In [ ]:
# GEOMETRY CHECK (read-only, prints only): hash-verify the accepted 19 geometry files against the manager lock, build the index with the
# FUNCTIONS above and run a full-extent page walk. Writes nothing; the API repeats the same verification at startup.
import hashlib

cfg = json.loads(CONFIG_PATH.read_text('utf-8'))
lock = json.loads((REPO / cfg['manager_lock']).read_text('utf-8'))


def _sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()


G = cfg['geometry']
for rel in (G['manifest'], G['geojson'], G['segments_csv']):
    assert _sha(REPO / rel) == lock['files'][rel], 'hash differs from the manager lock: ' + rel
man19 = json.loads((REPO / G['manifest']).read_text('utf-8'))
assert man19['status'] == 'complete'
assert man19['data_files_gitignored']['universe_geometry.geojson'] == lock['files'][G['geojson']]
assert man19['data_files_gitignored']['universe_segments.csv'] == lock['files'][G['segments_csv']]
IDX = build_geometry_index(json.loads((REPO / G['geojson']).read_text('utf-8')),
                           pd.read_csv(REPO / G['segments_csv'], dtype=str, keep_default_na=False), None, cfg['limits']['geometry'])
seen, cur = [], 0
while cur is not None:
    r = query_segments(IDX, tuple(cfg['limits']['geometry']['smoke_full_bbox']), int(cfg['limits']['geometry']['max_page_size']), cur, G['snapshot'])
    seen += [f['id'] for f in r['features']]
    cur = None if r['page']['next_cursor'] is None else int(r['page']['next_cursor'])
print(json.dumps({'segments': IDX['n'], 'extent': IDX['extent'], 'page_walk_ids': len(seen), 'unique': len(set(seen)),
                  'sorted': seen == sorted(seen), 'equals_index': seen == list(IDX['ids'])}, indent=1))
assert IDX['n'] == 3105 and seen == list(IDX['ids'])


## Outputs

None. This notebook only defines the pinned FUNCTIONS used by `backend/api/runtime.py` and prints a read-only geometry check. API evidence (OpenAPI, bounded smoke responses) is written by `backend/scripts/smoke_api_live.py` to a NEW ignored root such as `backend/rerun-api-019-001/`.
